In [1]:
import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ==========================================
# 1. Configuración
# ==========================================

ARCHIVO_DATOS = "correos.csv"
ARCHIVO_MODELO = "modelo_detector_spam.pkl"


# ==========================================
# 2. Crear datos de ejemplo si no existe CSV
# ==========================================

if not os.path.exists(ARCHIVO_DATOS):
    datos_ejemplo = pd.DataFrame({
        "texto": [
            "Ganaste un premio reclama tu dinero ahora",
            "Oferta exclusiva gana dinero fácilmente",
            "Haz clic aquí para recibir tu recompensa",
            "Eres el ganador de un teléfono gratis",
            "Última oportunidad para reclamar tu premio",
            "Compra ahora y recibe un descuento especial",
            "Reunión de trabajo programada para mañana",
            "Te envío el informe solicitado",
            "La clase comienza a las ocho de la mañana",
            "¿Puedes llamarme cuando tengas tiempo?",
            "Adjunto el documento de la reunión",
            "Recordatorio de la cita médica del viernes",
            "El proyecto fue aprobado por el cliente",
            "Nos vemos en la oficina esta tarde",
            "Gracias por tu mensaje, responderé pronto",
            "Recibe dinero gratis haciendo clic",
            "Has sido seleccionado para ganar un premio",
            "Oferta limitada, ingresa tus datos ahora",
            "Felicitaciones, ganaste un viaje gratis",
            "Obtén un préstamo sin requisitos"
        ],
        "etiqueta": [
            "spam",
            "spam",
            "spam",
            "spam",
            "spam",
            "spam",
            "no_spam",
            "no_spam",
            "no_spam",
            "no_spam",
            "no_spam",
            "no_spam",
            "no_spam",
            "no_spam",
            "no_spam",
            "spam",
            "spam",
            "spam",
            "spam",
            "spam"
        ]
    })

    datos_ejemplo.to_csv(ARCHIVO_DATOS, index=False, encoding="utf-8")

    print(f"Se creó el archivo de ejemplo: {ARCHIVO_DATOS}")


# ==========================================
# 3. Cargar los datos
# ==========================================

datos = pd.read_csv(ARCHIVO_DATOS, encoding="utf-8")

columnas_requeridas = ["texto", "etiqueta"]

if not all(columna in datos.columns for columna in columnas_requeridas):
    print("El archivo CSV debe contener estas columnas:")
    print("texto, etiqueta")
    exit()

datos = datos.dropna(subset=["texto", "etiqueta"])

X = datos["texto"].astype(str)
y = datos["etiqueta"].astype(str)


# ==========================================
# 4. Separar entrenamiento y prueba
# ==========================================

X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


# ==========================================
# 5. Crear el modelo Naive Bayes
# ==========================================

modelo = Pipeline([
    (
        "vectorizador",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=1
        )
    ),
    (
        "clasificador",
        MultinomialNB(alpha=1.0)
    )
])


# ==========================================
# 6. Entrenar el modelo
# ==========================================

modelo.fit(X_entrenamiento, y_entrenamiento)


# ==========================================
# 7. Evaluar el modelo
# ==========================================

predicciones = modelo.predict(X_prueba)

precision = accuracy_score(y_prueba, predicciones)
matriz = confusion_matrix(
    y_prueba,
    predicciones,
    labels=["no_spam", "spam"]
)

print("\nRESULTADOS DEL MODELO")
print("=====================")
print(f"Exactitud: {precision:.2%}")

print("\nReporte de clasificación:")
print(classification_report(y_prueba, predicciones))

print("Matriz de confusión:")
print(pd.DataFrame(
    matriz,
    index=["Real no_spam", "Real spam"],
    columns=["Predicho no_spam", "Predicho spam"]
))


# ==========================================
# 8. Guardar el modelo entrenado
# ==========================================

joblib.dump(modelo, ARCHIVO_MODELO)

print(f"\nModelo guardado como: {ARCHIVO_MODELO}")


# ==========================================
# 9. Clasificar correos escritos por el usuario
# ==========================================

print("\nDETECTOR DE SPAM")
print("================")
print("Escribe un correo para analizarlo.")
print("Escribe 'salir' para terminar.\n")

while True:
    correo = input("Correo: ").strip()

    if correo.lower() == "salir":
        print("Programa finalizado.")
        break

    if not correo:
        print("Debes escribir un correo.\n")
        continue

    resultado = modelo.predict([correo])[0]
    probabilidades = modelo.predict_proba([correo])[0]

    clases = modelo.named_steps["clasificador"].classes_
    probabilidades_por_clase = dict(zip(clases, probabilidades))

    probabilidad_spam = probabilidades_por_clase.get("spam", 0)
    probabilidad_no_spam = probabilidades_por_clase.get("no_spam", 0)

    print("\nRESULTADO")
    print("--------")
    print(f"Clasificación: {resultado}")
    print(f"Probabilidad de spam: {probabilidad_spam:.2%}")
    print(f"Probabilidad de no spam: {probabilidad_no_spam:.2%}")

    if resultado == "spam":
        print("Advertencia: este correo parece ser spam.")
    else:
        print("Este correo parece legítimo.")

    print()

Se creó el archivo de ejemplo: correos.csv

RESULTADOS DEL MODELO
Exactitud: 80.00%

Reporte de clasificación:
              precision    recall  f1-score   support

     no_spam       1.00      0.50      0.67         2
        spam       0.75      1.00      0.86         3

    accuracy                           0.80         5
   macro avg       0.88      0.75      0.76         5
weighted avg       0.85      0.80      0.78         5

Matriz de confusión:
              Predicho no_spam  Predicho spam
Real no_spam                 1              1
Real spam                    0              3

Modelo guardado como: modelo_detector_spam.pkl

DETECTOR DE SPAM
Escribe un correo para analizarlo.
Escribe 'salir' para terminar.

Correo: hola soy klim

RESULTADO
--------
Clasificación: spam
Probabilidad de spam: 53.33%
Probabilidad de no spam: 46.67%
Advertencia: este correo parece ser spam.

Correo: salir
Programa finalizado.
